# Colab T4 reproduction check: does a Colab T4 reproduce Kaggle T4 episodes exactly?

`noise_reachability/README.md`, "Colab T4 check". The frozen base and step 10's adapter N<t>_0 re-run 24 episodes
that step 10 ran on Kaggle T4 (tasks 0, 6, 7; step 12's harness states; seeds 16-19). If at least 11 of 12 match exactly
for each policy, Colab T4 can be used interchangeably with Kaggle T4 for paired comparisons.

**Inputs:** step 10's adapters N0_0, N6_0, N7_0 are read from the archive on Google Drive,
`MyDrive/Alienware_Research_archives/efficient_vla/noise-reachability-step-10/output/step10/runs/N<t>_0/adapter/`
(`tools/archive_kaggle_output.sh`). Step 6 checks their sha256 against the archive's manifest.

**Runtime: T4 GPU** (Runtime > Change runtime type). **Run all.** Nothing needs editing. About 1 h with the install
(estimate). If the runtime is lost, run all again: finished episodes are kept on Drive and skipped.

Download from step 10: `colab_t4_check.zip`.

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "55502717d4c9d60859d2e1416eb1700401d973d3"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The check of README.md ("Colab T4 check"); nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [0, 6, 7]
N_ACTION_STEPS = 10
HARNESS_STATE = {0: 8, 6: 8, 7: 33}   # step 12's harness episodes: this state x seeds 16-19 per task
HARNESS_SEEDS = "16-19"
POLICIES = {'base': ('base', 'base'), 'N0': ('lora', 'lora')}   # name: (--eval_policies, csv suffix); N0 is step 10's adapter N<t>_0
ARCHIVE = "/content/drive/MyDrive/Alienware_Research_archives/efficient_vla"   # tools/archive_kaggle_output.sh folders
ADAPTERS_IN = f"{ARCHIVE}/noise-reachability-step-10/output/step10/runs"         # step 10's N<t>_0/adapter/
ADAPTER_SHA256 = {0: {'adapter_config.json': '4f512e8d34de097adadb1d1e76324556a58d5a486dd98169929a772801a146fd', 'adapter_model.safetensors': 'aa163591bfc41a54da6d711908d828f7fe5b9a8d7d9f8895f1392c07e3e2d48d', 'run_config.json': 'faf912a80ab9fd77669633970d54325c0d51fbe8f6bad968e1e34d7448cd0f01'}, 6: {'adapter_config.json': '6e3e0f49f27fc0fef5a0d76761a971b2c5c4feec91ea81ee8cd41de657c69cc3', 'adapter_model.safetensors': '1eb3b7d5321c1a1d7455bc491920a2f2dfcf9542b81a91751163139c16eb0498', 'run_config.json': '16af5136a897ace291ef2b255db723f3614b81d318b7df73ee16977a1e7bed8c'}, 7: {'adapter_config.json': '33f536c16b513ab6673fa7106a9e38be80df456a246ccef0131f4d85aefdc91e', 'adapter_model.safetensors': '49c56ea1ea44cb6d4dd50c6362f4304531af410c355b03917c5cafa94bd00c40', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}}
DRIVE_DIR = "/content/drive/MyDrive/efficient_vla_runs/colab_t4_check"
OUT_ROOT = f"{DRIVE_DIR}/run"
STOP_AT_H = 4.0                    # no new work after this many hours of the session
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"Colab T4 check at {REPO_REF}: tasks {TASKS}, states {HARNESS_STATE}, seeds {HARNESS_SEEDS}, "
      f"policies {list(POLICIES)}, n_action_steps {N_ACTION_STEPS}, 24 episodes, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. GPU, RAM and Drive

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus and all("T4" in g[0] for g in gpus), f"GPU {gpus}: set the runtime type to T4 GPU (this check is about T4)"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2, int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
print(f"{len(gpus)} GPU(s) {gpus}, driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"{avail_gb:.0f} GB RAM available -> {JOBS} evaluation(s) at a time")

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{OUT_ROOT}/logs", exist_ok=True)

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment (the package versions of the Kaggle runs; several minutes)

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)
Path(OUT_ROOT, "environment.json").write_text(json.dumps(
    {"gpus": gpus, "cuda_driver": CUDA_DRIVER, "torch_build": TORCH_BUILD, "ram_available_gb": round(avail_gb, 1)},
    indent=1) + "\n")

## 6. Commit of this run and step 10's adapters (sha256 against the archive)

In [ ]:
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the check started at commit {started_with}, this session cloned {head}")
with open(first, "a") as f:
    f.write(head + "\n")
for t, files in ADAPTER_SHA256.items():
    for name, want in files.items():
        p = Path(ADAPTERS_IN, f"N{t}_0", "adapter", name)
        assert p.exists(), f"upload step 10's adapter N{t}_0 to {p.parent}"
        got = hashlib.sha256(p.read_bytes()).hexdigest()
        assert got == want, f"{p}: sha256 {got}, expected step 10's {want}"
    # self_filtered_bc refuses an adapter trained with another checkpoint path; Kaggle and Colab both run as root
    trained_with = json.loads(Path(ADAPTERS_IN, f"N{t}_0", "adapter", "run_config.json").read_text())["policy_path"]
    assert trained_with == POLICY_PATH, f"N{t}_0 was trained with {trained_with}, this session has {POLICY_PATH}"
print("adapters N0_0, N6_0, N7_0: sha256 equal to step 10's archive, trained with this checkpoint path")

## 7. Helpers: jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 10, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






MAIN = f"{REPO}/self_filtered_bc/self_filtered_bc.py"


def sfbc(task):
    return f"cd {REPO} && {PY} -u {MAIN} --task_id {task} --n_action_steps {N_ACTION_STEPS} --policy_path {POLICY_PATH}"

## 8. The 24 episodes: base and N<t>_0 on step 12's harness episodes (one process per task and policy)

In [ ]:
jobs = {}
for t in TASKS:
    for policy, (flag, suffix) in POLICIES.items():
        out = f"{OUT_ROOT}/task_{t}/{policy}"
        if len(read_csv(f"{out}/per_episode_{suffix}.csv")) < n_range(HARNESS_SEEDS):
            lora = f" --lora_dir {ADAPTERS_IN}/N{t}_0/adapter" if flag == "lora" else ""
            jobs[f"task{t}_{policy}"] = (f"{sfbc(t)} --stage eval --eval_policies {flag}{lora} --eval_seeds {HARNESS_SEEDS} "
                                         f"--eval_init_states {HARNESS_STATE[t]} --eval_output_dir {out} "
                                         f"--max_episodes_per_process 10", 0)
print(f"{len(jobs)} of {len(TASKS) * len(POLICIES)} jobs to run", flush=True)
codes = run_jobs(jobs, max_parallel=JOBS)
if failed(codes):
    fail(f"evaluation failed: {failed(codes)}; see {OUT_ROOT}/logs")

## 9. The comparison with step 10's Kaggle T4 episodes

In [ ]:
if not FAILURE:
    r = sh(f"cd {REPO} && python3 -S noise_reachability/analyze_colab_t4_check.py --root {OUT_ROOT} "
           f"--out {OUT_ROOT}/colab_t4_check.txt", check=False)
    if r.returncode != 0:
        fail("the analysis refused the run: read its message above")

## 10. Download

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /content/colab_t4_check.zip . -x '*.pyc' -x '*.tmp*'")
from google.colab import files
files.download("/content/colab_t4_check.zip")
if FAILURE:
    raise RuntimeError(FAILURE)